In [2]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

from src.parsing import get_html, parse_course_page, get_target_courses

In [3]:
from src.parsing import BASE

In [4]:
targets = [
    "Mathematics Advanced",
    "Mathematics Extension 1",
    "Mathematics Extension 2",
    "English Advanced",
    "English Standard",
    "Biology",
    "Chemistry",
    "Physics",
    "Legal Studies",
]

all_rows = []
for year in [2021, 2022, 2023, 2024, 2025]:
    tdf = get_target_courses(year, targets)      # note: pass targets now
    for c in tdf.itertuples():
        html = get_html(f"{BASE}/BDHSC_{year}_12_{c.course_code}.html",
                        f"BDHSC_{year}_{c.course_code}.html")
        all_rows.extend(parse_course_page(html, year, c.course_name, c.course_code))

print(len(all_rows))   # expect 250 — and instant, because it's all cached

250


In [5]:
import pandas as pd

df = pd.read_csv("../data/interim/hsc_bands_raw.csv")
print(df.shape)
df.head()

(250, 6)


,band,percentage,candidature,year,course_name,course_code
0,6,7.16,18712,2021,Biology,15030
1,5,24.14,18712,2021,Biology,15030
2,4,34.79,18712,2021,Biology,15030
3,3,25.15,18712,2021,Biology,15030
4,2,6.68,18712,2021,Biology,15030


In [6]:
df.dtypes

band               str
percentage     float64
candidature      int64
year             int64
course_name        str
course_code      int64
dtype: object

In [7]:
band_sums = df.groupby(["course_name", "year"])["percentage"].sum()
band_sums

course_name              year
Biology                  2021     99.97
                         2022     99.97
                         2023    100.01
                         2024     99.99
                         2025     99.99
Chemistry                2021     99.96
                         2022     99.96
                         2023     99.99
                         2024    100.01
                         2025    100.00
English Advanced         2021     99.98
                         2022     99.97
                         2023    100.00
                         2024    100.00
                         2025    100.01
English Standard         2021     99.97
                         2022     99.97
                         2023    100.00
                         2024    100.00
                         2025     99.99
Legal Studies            2021     99.96
                         2022     99.98
                         2023     99.99
                         2024    100.01
          

In [8]:
# TODO: show only the course-years whose sum is NOT within 0.5 of 100
#   hint: band_sums is a Series of numbers. Build a boolean mask like
#         (band_sums < 99.5) | (band_sums > 100.5)  and use it to filter band_sums

band_sums[(band_sums < 99.5) | (band_sums > 100.5)]

Series([], Name: percentage, dtype: float64)

In [9]:
band_sums.describe()          # or: print(band_sums.min(), band_sums.max())

count     45.000000
mean      99.987778
std        0.014754
min       99.960000
25%       99.980000
50%       99.990000
75%      100.000000
max      100.010000
Name: percentage, dtype: float64

In [10]:
df.isna().sum()               # any missing values? (expect all zeros)
df.groupby(["course_name", "year"]).size()    # rows per course-year

course_name              year
Biology                  2021    6
                         2022    6
                         2023    6
                         2024    6
                         2025    6
Chemistry                2021    6
                         2022    6
                         2023    6
                         2024    6
                         2025    6
English Advanced         2021    6
                         2022    6
                         2023    6
                         2024    6
                         2025    6
English Standard         2021    6
                         2022    6
                         2023    6
                         2024    6
                         2025    6
Legal Studies            2021    6
                         2022    6
                         2023    6
                         2024    6
                         2025    6
Mathematics Advanced     2021    6
                         2022    6
                         

In [11]:
import numpy as np
# TODO: create df["band_scale"]:
#   "E1-E4" where band starts with "E", otherwise "1-6"
#   hint: np.where(df["band"].str.startswith("E"), "E1-E4", "1-6")

df["band_scale"] = np.where(df["band"].str.startswith("E"), "E1-E4", "1-6")
df

,band,percentage,candidature,year,course_name,course_code,band_scale
0,6,7.16,18712,2021,Biology,15030,1-6
1,5,24.14,18712,2021,Biology,15030,1-6
2,4,34.79,18712,2021,Biology,15030,1-6
3,3,25.15,18712,2021,Biology,15030,1-6
4,2,6.68,18712,2021,Biology,15030,1-6
...,...,...,...,...,...,...,...
245,5,25.19,8817,2025,Physics,15330,1-6
246,4,25.75,8817,2025,Physics,15330,1-6
247,3,21.21,8817,2025,Physics,15330,1-6
248,2,13.81,8817,2025,Physics,15330,1-6


In [12]:
# TODO: create df["count"] = percentage/100 * candidature, rounded to a whole number
#   hint: (df["percentage"] / 100 * df["candidature"]).round().astype(int)
df["count"] = (df["percentage"]/100 * df["candidature"]).round().astype(int)
df

,band,percentage,candidature,year,course_name,course_code,band_scale,count
0,6,7.16,18712,2021,Biology,15030,1-6,1340
1,5,24.14,18712,2021,Biology,15030,1-6,4517
2,4,34.79,18712,2021,Biology,15030,1-6,6510
3,3,25.15,18712,2021,Biology,15030,1-6,4706
4,2,6.68,18712,2021,Biology,15030,1-6,1250
...,...,...,...,...,...,...,...,...
245,5,25.19,8817,2025,Physics,15330,1-6,2221
246,4,25.75,8817,2025,Physics,15330,1-6,2270
247,3,21.21,8817,2025,Physics,15330,1-6,1870
248,2,13.81,8817,2025,Physics,15330,1-6,1218


In [13]:
df.head()

,band,percentage,candidature,year,course_name,course_code,band_scale,count
0,6,7.16,18712,2021,Biology,15030,1-6,1340
1,5,24.14,18712,2021,Biology,15030,1-6,4517
2,4,34.79,18712,2021,Biology,15030,1-6,6510
3,3,25.15,18712,2021,Biology,15030,1-6,4706
4,2,6.68,18712,2021,Biology,15030,1-6,1250


In [14]:
band_order = ["1", "2", "3", "4", "5", "6", "E1", "E2", "E3", "E4"]
df["band"] = pd.Categorical(df["band"], categories=band_order, ordered=True)
df["band"]      # dtype should now say: category, and show the ordering

0      6
1      5
2      4
3      3
4      2
      ..
245    5
246    4
247    3
248    2
249    1
Name: band, Length: 250, dtype: category
Categories (10, str): ['1' < '2' < '3' < '4' ... 'E1' < 'E2' < 'E3' < 'E4']

In [15]:
df.to_csv("../data/processed/hsc_bands_clean.csv", index=False)

In [16]:
print(df["band"].cat.ordered)        # True
print(df["band"].cat.categories)     # ['1','2',...,'6','E1',...,'E4'] in order
df.dtypes                            # band should now say 'category'

True
Index(['1', '2', '3', '4', '5', '6', 'E1', 'E2', 'E3', 'E4'], dtype='str')


band           category
percentage      float64
candidature       int64
year              int64
course_name         str
course_code       int64
band_scale          str
count             int64
dtype: object